# Step-response baseline synchrony grid summary

Scalar-first analysis for step-response simulation batches. This notebook builds or reloads cached repeat-level tables, optionally recomputes missing baseline synchrony from saved spike trains, plots LF-row alpha x beta baseline heatmaps, and can also plot post-step net-input heatmaps.

In [ ]:
from pathlib import Path

import pandas as pd

from step_response_sync_summary import (
    build_post_step_net_input_sync_heatmap_table,
    build_or_load_step_response_tables,
    collect_step_response_mean_trace_table,
    load_post_step_net_input_heatmap_table,
    plot_baseline_sync_heatmaps,
    plot_post_step_net_input_sync_heatmaps,
    plot_step_reactivity_summary,
    save_post_step_net_input_heatmap_table,
)

In [ ]:

# Main batch folder. Update this if you want to summarize another step-response batch.
BATCH_ROOT = Path(r"$$$$_sim_batches_STEP_RESPONSE_with_repeats_V1")
OUTPUT_SUBDIR_NAME = "step_response_summary"

# Cache controls.
FORCE_REBUILD_REPEAT_TABLE = False
RECOMPUTE_BASELINE_SYNC_IF_MISSING = True
POOL_STEP_AMPLITUDES_FOR_BASELINE_SYNC = True
# For paired-step batches, set this to True and rebuild once to avoid counting
# matched nonzero-step runs as extra baseline stochastic realizations.
USE_ZERO_STEP_REFERENCE_ONLY_FOR_BASELINE_SYNC = False

# Use an integer, e.g. 20, for a quick smoke run. Use None for the full batch.
MAX_RUNS_FOR_DEBUG = None
EXTRACTION_N_JOBS = 1  # Increase after a debug run if baseline sync needs recomputation for many files.

ALPHA_BETA_HEATMAP_BINNING = "auto"  # "auto", "exact", or "quantile"
ALPHA_BETA_HEATMAP_QUANTILE_BINS = 4

HEATMAP_STYLE = {
    "figsize_per_row": (10.5, 3.3),
    "mean_cmap": "viridis",
    "sd_cmap": "magma",
    "across_run_cv_cmap": "cividis",
    "mean_vmin": None,
    "mean_vmax": None,
    "sd_vmin": None,
    "sd_vmax": None,
    "across_run_cv_vmin": None,
    "across_run_cv_vmax": None,
    # Optional percentile limits, e.g. (2, 98). Manual vmin/vmax override these.
    "mean_color_percentiles": None,
    "sd_color_percentiles": None,
    "across_run_cv_color_percentiles": None,
    "annotate": True,
    "annotation_fmt": ".3f",
    "show_cell_n": True,
    "annotation_fontsize": 7.0,
    "title_fontsize": 10.0,
    "axis_label_fontsize": 9.0,
    "tick_label_fontsize": 8.0,
    "colorbar_label_fontsize": 8.0,
    "suptitle_fontsize": 13.0,
    "title": "Baseline synchrony across alpha/beta input amplitudes",
}

PLOT_POST_STEP_NET_INPUT_HEATMAPS = True
FORCE_REBUILD_POST_STEP_NET_INPUT_HEATMAP_TABLE = False
POST_STEP_NET_INPUT_HEATMAP_STYLE = {
    "filename_base": "post_step_sync_alpha_beta_by_net_input",
    "figsize_per_row": (10.5, 3.3),
    "mean_cmap": "viridis",
    "sd_cmap": "magma",
    "across_run_cv_cmap": "cividis",
    "mean_vmin": None,
    "mean_vmax": None,
    "sd_vmin": None,
    "sd_vmax": None,
    "across_run_cv_vmin": None,
    "across_run_cv_vmax": None,
    # Optional percentile limits, e.g. (2, 98). Manual vmin/vmax override these.
    "mean_color_percentiles": None,
    "sd_color_percentiles": None,
    "across_run_cv_color_percentiles": (5, 95), # None,
    "annotate": True,
    "annotation_fmt": ".3f",
    "show_cell_n": True,
    "annotation_fontsize": 7.0,
    "title_fontsize": 10.0,
    "axis_label_fontsize": 9.0,
    "tick_label_fontsize": 8.0,
    "colorbar_label_fontsize": 8.0,
    "suptitle_fontsize": 13.0,
    "title": "Post-step synchrony by post-step net input and alpha/beta amplitudes",
}

PLOT_STEP_REACTIVITY_SUMMARY = True
STEP_REACTIVITY_DATA_SOURCE = "mean_trace"  # "repeats" or "mean_trace"
STEP_REACTIVITY_METRIC = "shape_t50" # accepted: "gain", "linear_slope", "shape_tau", "shape_t50", "shape_t90", "response_50ms", "response_150ms", "xcorr_lag", "xcorr_lag_sign_corrected"
STEP_REACTIVITY_SYNC_SOURCE = "local_matched"  # accepted: "baseline", "baseline_sd", "pre_actual", "pre_actual_sd", "pre_matched", "pre_matched_sd", "local_matched", "local_matched_sd", "post_actual", "post_actual_sd"
STEP_REACTIVITY_STYLE = {
    "step_amplitudes_uA": None, # [10.0], # None,  # e.g. [-10.0, 10.0, 25.0]; None uses all nonzero steps.
    "figsize_per_step": (20, 7),
    "cmap": "viridis", # "coolwarm",
    "point_size": 28.0,
    "point_alpha": 0.82,
    "point_edgecolor": "none",
    "show_regression_lines": True,
    "regression_line_color": "black",
    "regression_line_lw": 1.5,
    "regression_line_alpha": 0.45,
    "regression_line_ls": "-",
    "wspace": 0.4,
    "hspace": 0.2,
    "metric_vmin": None,
    "metric_vmax": None,
    "metric_color_percentiles": None,
    "metric_color_scale": "auto",  # "auto" uses data min/max; "symmetric" centers colors on 0 for positive/negative steps.
    "center_color_scale_on_zero": False,  # compatibility alias; overridden by metric_color_scale when set.
    "alpha_xlim": None,
    "beta_xlim": None,
    "sync_xlim": None,
    "metric_ylim": None,
    "axis_pad_fraction": 0.05,
    "title_fontsize": 10.0,
    "axis_label_fontsize": 9.0,
    "annotation_fontsize": 8.0,
    "suptitle_fontsize": 13.0,
    "colorbar_label_fontsize": 9.0,
    "require_matched_step_effect": True,
    "save_plot_table": True,
    "title": None,
}

if not BATCH_ROOT.exists():
    candidates = sorted(p for p in Path('.').iterdir() if p.is_dir() and 'STEP_RESPONSE' in p.name)
    raise FileNotFoundError(
        f"BATCH_ROOT does not exist: {BATCH_ROOT}. Available STEP_RESPONSE folders: {[p.name for p in candidates]}"
    )

print(f"Batch root: {BATCH_ROOT.resolve()}")

In [ ]:
result = build_or_load_step_response_tables(
    BATCH_ROOT,
    output_subdir_name=OUTPUT_SUBDIR_NAME,
    force_rebuild=FORCE_REBUILD_REPEAT_TABLE,
    recompute_baseline_sync_if_missing=RECOMPUTE_BASELINE_SYNC_IF_MISSING,
    pool_step_amplitudes=POOL_STEP_AMPLITUDES_FOR_BASELINE_SYNC,
    use_zero_step_reference_only=USE_ZERO_STEP_REFERENCE_ONLY_FOR_BASELINE_SYNC,
    alpha_beta_heatmap_binning=ALPHA_BETA_HEATMAP_BINNING,
    alpha_beta_heatmap_quantile_bins=ALPHA_BETA_HEATMAP_QUANTILE_BINS,
    max_runs=MAX_RUNS_FOR_DEBUG,
    n_jobs=EXTRACTION_N_JOBS,
)

repeat_df = result["repeat_df"]
parameter_set_df = result["parameter_set_df"]
heatmap_df = result["heatmap_df"]
alpha_beta_binning_metadata = result["alpha_beta_binning_metadata"]
output_dir = result["output_dir"]
priors = result["priors"]

print(f"Cache source: {result['cache_source']}")
print(f"Output dir: {output_dir.resolve()}")
print(f"Priors: {priors['priors_path']}")
print(f"Repeat rows: {len(repeat_df)}")
print(f"Parameter-set rows: {len(parameter_set_df)}")
print(f"Heatmap rows: {len(heatmap_df)}")
print(f"Alpha/beta heatmap binning: {alpha_beta_binning_metadata['effective_binning']} ({alpha_beta_binning_metadata['slug']})")

display(
    repeat_df["baseline_sync_source"].value_counts(dropna=False).rename_axis("baseline_sync_source").to_frame("n_runs")
)
display(
    heatmap_df.head()
)


In [ ]:
post_step_net_input_heatmap_df = None
post_step_net_input_heatmap_cache_source = "disabled"
post_step_net_input_heatmap_paths = None

if PLOT_POST_STEP_NET_INPUT_HEATMAPS:
    cached_post_step_heatmap = None if FORCE_REBUILD_POST_STEP_NET_INPUT_HEATMAP_TABLE else load_post_step_net_input_heatmap_table(output_dir, alpha_beta_binning_metadata)
    if cached_post_step_heatmap is None:
        post_step_net_input_heatmap_df = build_post_step_net_input_sync_heatmap_table(
            repeat_df,
            alpha_beta_binning_metadata=alpha_beta_binning_metadata,
        )
        post_step_net_input_heatmap_paths = save_post_step_net_input_heatmap_table(
            output_dir,
            post_step_net_input_heatmap_df,
            alpha_beta_binning_metadata,
        )
        post_step_net_input_heatmap_cache_source = "rebuilt"
    else:
        post_step_net_input_heatmap_df = cached_post_step_heatmap
        post_step_net_input_heatmap_cache_source = "cached"

    print(f"Post-step net-input heatmap cache source: {post_step_net_input_heatmap_cache_source}")
    print(f"Post-step net-input heatmap rows: {len(post_step_net_input_heatmap_df)}")
    if post_step_net_input_heatmap_paths is not None:
        print(f"Saved post-step heatmap table PKL: {post_step_net_input_heatmap_paths['pkl_path']}")
        print(f"Saved post-step heatmap table CSV: {post_step_net_input_heatmap_paths['csv_path']}")
    display(post_step_net_input_heatmap_df.head())


In [ ]:
plot_result = plot_baseline_sync_heatmaps(
    heatmap_df,
    output_dir=output_dir,
    **HEATMAP_STYLE,
)

print(f"Saved PNG: {plot_result['png_path']}")
print(f"Saved PDF: {plot_result['pdf_path']}")

In [ ]:
if PLOT_POST_STEP_NET_INPUT_HEATMAPS and post_step_net_input_heatmap_df is not None:
    post_step_plot_result = plot_post_step_net_input_sync_heatmaps(
        post_step_net_input_heatmap_df,
        output_dir=output_dir,
        **POST_STEP_NET_INPUT_HEATMAP_STYLE,
    )
    for figure_info in post_step_plot_result["figures"]:
        print(
            f"Saved LF={figure_info['lf_target_sd'] / 1000:g} uA PNG: {figure_info['png_path']}"
        )
        print(
            f"Saved LF={figure_info['lf_target_sd'] / 1000:g} uA PDF: {figure_info['pdf_path']}"
        )


In [ ]:
if PLOT_STEP_REACTIVITY_SUMMARY:
    if STEP_REACTIVITY_DATA_SOURCE == "repeats":
        reactivity_input_df = repeat_df
    elif STEP_REACTIVITY_DATA_SOURCE == "mean_trace":
        reactivity_input_df = collect_step_response_mean_trace_table(
            BATCH_ROOT,
            parameter_set_df=parameter_set_df,
            priors=priors,
            max_runs=MAX_RUNS_FOR_DEBUG,
        )
        mean_trace_table_pkl = output_dir / "step_response_mean_trace_reactivity_table.pkl"
        mean_trace_table_csv = output_dir / "step_response_mean_trace_reactivity_table.csv"
        reactivity_input_df.to_pickle(mean_trace_table_pkl)
        reactivity_input_df.to_csv(mean_trace_table_csv, index=False)
        print(f"Saved mean-trace reactivity source table PKL: {mean_trace_table_pkl}")
        print(f"Saved mean-trace reactivity source table CSV: {mean_trace_table_csv}")
    else:
        raise ValueError("STEP_REACTIVITY_DATA_SOURCE must be 'repeats' or 'mean_trace'")
    reactivity_plot_result = plot_step_reactivity_summary(
        reactivity_input_df,
        output_dir=output_dir,
        reactivity_metric=STEP_REACTIVITY_METRIC,
        sync_source=STEP_REACTIVITY_SYNC_SOURCE,
        **STEP_REACTIVITY_STYLE,
    )
    print(f"Saved reactivity PNG: {reactivity_plot_result['png_path']}")
    print(f"Saved reactivity PDF: {reactivity_plot_result['pdf_path']}")
    if 'table_pkl_path' in reactivity_plot_result:
        print(f"Saved plotted-row table PKL: {reactivity_plot_result['table_pkl_path']}")
        print(f"Saved plotted-row table CSV: {reactivity_plot_result['table_csv_path']}")
    print(
        f"Metric color scale: {reactivity_plot_result['metric_vmin']:.4g} to {reactivity_plot_result['metric_vmax']:.4g}"
    )
    display(reactivity_plot_result["plot_df"].head())
